# 08. Querying Azure AI Search Directly

This script steps outside the Agent Service entirely — no `azure-ai-projects`, no `PromptAgentDefinition`. It
uses `azure.search.documents.SearchClient` to run a plain **keyword (full-text) search** against an existing
Azure AI Search index (`"rag-1782198581571"` — an auto-generated index name, the kind Foundry's "Add your
data"/ingestion tooling produces when it builds a search index from uploaded documents). This is the
foundational building block for the manual RAG pattern shown in `09_customer_rag_agent.py` /
`10_customer_rag_client.py`: before grounding an *agent's* answer in search results, you need to understand
how to query the search index on its own.

**Difficulty:** Beginner


## Prerequisites

**Pip packages:**
- `azure-search-documents` — **not currently in the repo's root `requirements.txt`**, install with:
  ```bash
  pip install azure-search-documents
  ```
- `azure-identity` (already in root `requirements.txt`)

**Azure resource(s) required:**
- An **Azure AI Search** service, with an index already populated (this course's example service is
  `cloudxeus-search01`, index `rag-1782198581571`).

**Environment variables** (add to the root `.env`):
- `AZURE_SEARCH_ENDPOINT` — e.g. `https://<your-search-service>.search.windows.net`
- `AZURE_SEARCH_INDEX_NAME` — the index to query
- (Optional alternative) `AZURE_SEARCH_API_KEY` — only needed if you switch from Entra ID auth to key-based
  auth (see the Alternatives note below); the original script uses `DefaultAzureCredential`, so no key is
  strictly required if your identity has the right RBAC role on the search service (e.g. *Search Index Data
  Reader*).


## What You'll Learn

- How to construct a `SearchClient` for a specific index using Entra ID auth
- How `search_client.search(search_text=..., select=[...])` performs a keyword/full-text query
- What fields typical RAG-oriented Azure AI Search indexes expose (`chunk`, `title`, `@search.score`)
- Why this raw search building block matters before layering an agent's grounded-prompt logic on top of it
  (`09`/`10`)


### Constructing the search client and querying

`SearchClient` needs three things: the search service's `endpoint`, the specific `index_name` to query, and a
`credential`. Here it's `DefaultAzureCredential()` — the same Entra ID pattern used for Foundry project access
elsewhere in this section, but note it's a **separate Azure resource** (Azure AI Search, not Azure AI Foundry)
with its own RBAC role requirements. `search_client.search(search_text="refund", select=["chunk", "title"])`
runs a keyword query for "refund" and asks the service to return only the `chunk` and `title` fields per
matching document (plus the always-included `@search.score` relevance score).

- **💡 Exam tip:** `search_text` alone (no `vector_queries`) performs classic **keyword/full-text search**
  (BM25-style ranking) — this is distinct from **vector search** (`VectorizableTextQuery`, seen in
  `10_customer_rag_client.py`) and from **hybrid search** (using both together). AI-102 expects you to
  distinguish these three retrieval modes.
- **🔄 Alternatives:** `AzureKeyCredential(api_key)` instead of `DefaultAzureCredential()` — simpler to set up
  for a quick local test (just needs an admin or query API key from the search service), but Entra ID/RBAC is
  the recommended approach for production since it avoids managing a long-lived secret.


In [ ]:
import os

from dotenv import load_dotenv
from azure.identity import DefaultAzureCredential
from azure.search.documents import SearchClient

load_dotenv()

SEARCH_ENDPOINT = os.getenv("AZURE_SEARCH_ENDPOINT")
INDEX_NAME = os.getenv("AZURE_SEARCH_INDEX_NAME", "rag-1782198581571")

search_client = SearchClient(
    endpoint=SEARCH_ENDPOINT,
    index_name=INDEX_NAME,
    credential=DefaultAzureCredential(),
)

results = search_client.search(
    search_text="refund",
    select=["chunk", "title"],
)

for result in results:
    print(f"Score:  {result['@search.score']:.4f}")
    print(f"Source: {result['title']}")
    print(f"Text:   {result['chunk']}")
    print("---")


## Summary

This notebook runs a plain keyword search for "refund" against an Azure AI Search index, printing each
matching chunk's relevance score, source title, and text. This is exactly the kind of query that
`10_customer_rag_client.py` later performs (as part of a hybrid vector+keyword query) to gather grounding text
before handing it to a Foundry agent — seeing it here in isolation makes the retrieval step easy to reason
about on its own.


## Try It Yourself

1. **Easy:** Change `search_text` to a different query term relevant to your own index's content and observe
   how the returned `@search.score` values change.
2. **Intermediate:** Add `top=5` to the `search()` call to limit results, and add `"parent_id"` to `select` to
   see the document's parent/source identifier alongside each chunk.
3. **Advanced:** Look up `search_client.search`'s `filter` parameter (OData syntax) and add a filter restricting
   results to a specific `title`, combining keyword search with a metadata filter.
